# Sensor Reconstruction using a Masked Graph Autoencoder

## Objective

The objective of this project is to reconstruct the value of a missing vibration sensor using the measurements of the remaining sensors **at the same timestamp**.

Unlike conventional time-series forecasting methods such as LSTMs or ARIMA, this approach **does not use historical observations**. Each timestamp is treated as an independent sample because vibration events can be highly irregular and unpredictable.

Instead of learning temporal dependencies, the model learns the **relationships between sensors** represented as a graph.

## Why a Graph Neural Network?

Traditional machine learning models consider the input as a simple feature vector:

$$ x = [x_1,x_2,\ldots,x_n] $$

where every feature is treated independently.

However, industrial sensors are physically connected.

For example,

Motor → Pump → Pressure → Temperature

Changes in one component influence the others through the physical behavior of the machine.

Graph Neural Networks (GNNs) explicitly model these relationships by representing each sensor as a **node** and the physical interactions between sensors as **edges**.

Instead of learning from isolated variables, the network propagates information between connected sensors, allowing each sensor to benefit from the information carried by its neighbors.

## Why a Graph Autoencoder?

A Graph Autoencoder (GAE) is a neural network composed of two parts:

- **Encoder**
- **Decoder**

The encoder compresses the graph into a compact latent representation.

The decoder reconstructs the original information from this compressed representation.

The general architecture is

            Input Graph
                    │
                    ▼
                Encoder
                    │
                    ▼
        Latent Representation
                    │
                    ▼
                Decoder
                    │
                    ▼
        Reconstructed Graph

The encoder learns the most informative representation of the graph while the decoder attempts to reconstruct the original data as accurately as possible.

## Why not use a classical Graph Autoencoder?

Classical Graph Autoencoders were originally developed for **link prediction**.

Their objective is to reconstruct the **graph structure (adjacency matrix)** by predicting whether an edge exists between two nodes.

For example,

Motor ───── Pump

↓

Predict whether

Motor ───── Pressure

should also be connected.

This is not the objective of this project.

The graph topology of the industrial system is already known.

Instead, the objective is to reconstruct **missing sensor measurements** while keeping the graph structure fixed.

## Proposed Architecture

The proposed architecture is a **Masked Graph Autoencoder (MGAE)**.

Instead of removing graph connections, one or more sensor values are randomly hidden during training.

For example,

Original measurements

```m
Motor PV = 2.45
Pump PV = 3.82
Pressure = 5.31
Temperature = 27.8
```

Training input

```m
Motor PV = ?
Pump PV = 3.82
Pressure = 5.31
Temperature = 27.8
```

Target
```m
Motor PV = 2.45
```

The network learns to estimate the missing sensor using only the information available from the remaining sensors.

This training strategy is known as **self-supervised learning**, since the training targets are automatically generated by masking existing sensor values.

The proposed model consists of four main stages.

                     Sensor Graph
                          │
                          ▼
                Random Node Masking
                          │
                          ▼
                   Graph Encoder
                          │
                          ▼
               Latent Representation
                          │
                          ▼
                   Graph Decoder
                          │
                          ▼
          Reconstructed Sensor Values

Only the masked sensors contribute to the reconstruction loss during training.

## Why this approach?

This methodology presents several advantages for industrial sensor reconstruction.

- No historical data is required.
- Each timestamp is processed independently.
- Physical relationships between sensors are incorporated through the graph structure.
- The model can reconstruct missing sensors without requiring manually labeled data.
- The same architecture can naturally handle one or multiple missing sensors.
- The learned latent representation captures complex nonlinear relationships that are difficult to model analytically.

## Load the data

This section loads the preprocessed sensor dataset and prepares the feature matrix for the model.
The timestamp column is removed because each row is treated independently and the model only uses the sensor values.
The features are standardized so the GNN can train more reliably.

In [8]:
import pandas as pd

df = pd.read_csv("../data/processed/synthetic_A_sensor.csv", decimal=",")

df.head()

,Timestamp,A_ACR_Mot.PV,A_ACR_Mot.SV,A_ACR_Mot.TV,A_ACR_Pmp.PV,A_ACR_Pmp.SV,A_ACR_Pmp.TV,A_Pres.PV,A_Temp.PV,Barometer,Temperature
0,2024-10-30 10:30:00+00:00,0.0021952891222139,3.5014076838214776,45.36849389797249,0.0016873006212543,13.74421058490841,38.154255629422,19.572690575396347,38.71038998550927,1020.2599781742224,23.126361212339017
1,2024-10-30 10:30:01+00:00,0.0022601265620296,3.100834970846942,45.50726937042017,0.001753513722955,13.288506040390914,38.27625309953265,19.600693610180542,38.98924015321537,1020.270705360722,23.09890106811812
2,2024-10-30 10:30:02+00:00,0.0022242005053888,3.128098896636948,45.40413176568267,0.0016658460692344,13.25976251321527,38.18389952774681,18.741180764856,39.137689730862334,1020.224881762647,23.24650044224058
3,2024-10-30 10:30:03+00:00,0.0022306119084681,2.9065401001130464,45.57047702431612,0.00132517797206,13.637371424414145,38.174162349931954,15.75514358484774,39.90589732051072,1020.212360839398,23.12271353956653
4,2024-10-30 10:30:04+00:00,0.0021680275103897,3.0283288895173675,45.82223408997572,0.0014608856266696,12.50887871553521,38.03401954016153,20.58975988471463,38.95574045081952,1020.2636540080632,23.20401358642702


In [9]:
features = df.drop(columns=["Timestamp"])

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = scaler.fit_transform(features)
X.shape

(21600, 10)

Here we define the graph structure of the sensor network.
Each sensor becomes a node and physical relationships between sensors are represented as bidirectional edges.
This graph is fixed and reused for every timestamp.

## Create the graph

In [5]:
sensor_names = [
    "A_ACR_Mot.PV",
    "A_ACR_Mot.SV",
    "A_ACR_Mot.TV",
    "A_ACR_Pmp.PV",
    "A_ACR_Pmp.SV",
    "A_ACR_Pmp.TV",
    "A_Pres.PV",
    "A_Temp.PV",
    "Barometer",
    "Temperature",
]

Each row of sensor measurements is converted into a graph object with node features and the shared edge connectivity.
The resulting list of graphs is the input dataset for the GNN.

In [6]:
import torch

edge_index = torch.tensor([
    [0,1],
    [1,0],

    [0,2],
    [2,0],

    [0,3],
    [3,0],

    [3,4],
    [4,3],

    [3,5],
    [5,3],

    [3,6],
    [6,3],

    [6,7],
    [7,6],

    [7,8],
    [8,7],

    [7,9],
    [9,7]
], dtype=torch.long).t()

## Convert every row into a graph


This section converts each timestamp into a graph data object suitable for PyTorch Geometric.
Every sensor reading becomes a node feature, while the shared edge structure captures the sensor relationships.


In [7]:
from torch_geometric.data import Data

graphs = []

for row in X:

    x = torch.tensor(row, dtype=torch.float32).view(-1,1)

    graph = Data(
        x=x,
        edge_index=edge_index
    )

    graphs.append(graph)

c:\Users\intel\industry\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Random Mapping

This helper function randomly selects one sensor node to hide and uses the original value as the reconstruction target.
The model learns to infer the missing reading from the remaining connected sensor nodes.

In [ ]:
import random

def mask_graph(data):

    data = data.clone()

    target = random.randint(0,9)

    y = data.x.clone()

    data.x[target] = 0

    return data,y,target

## Encoder

The encoder applies graph convolution layers to propagate information across the sensor graph.
It produces a latent embedding for each node that summarizes both the node value and its neighbors.

In [ ]:
import torch.nn as nn
from torch_geometric.nn import GCNConv

class Encoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(1,32)

        self.conv2 = GCNConv(32,16)

    def forward(self,x,edge_index):

        x=self.conv1(x,edge_index)

        x=torch.relu(x)

        x=self.conv2(x,edge_index)

        return x

## Decoder

The decoder maps each node embedding back to a sensor value.
It reconstructs the original measurement for every node, but we only measure loss on the masked node.

In [ ]:
class Decoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc=nn.Sequential(

            nn.Linear(16,32),

            nn.ReLU(),

            nn.Linear(32,1)

        )

    def forward(self,z):

        return self.fc(z)

## Model

The full model combines the encoder and decoder into a graph autoencoder.
It takes a masked graph as input and returns reconstructed sensor values for every node.

In [ ]:
class GraphAutoEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder=Encoder()

        self.decoder=Decoder()

    def forward(self,data):

        z=self.encoder(data.x,data.edge_index)

        out=self.decoder(z)

        return out

## Loss

The loss is defined only on the masked node's predicted value.
By comparing the reconstructed value with the original measurement, the model learns to recover missing sensor readings.

In [ ]:
criterion = nn.MSELoss()

prediction = model(masked_graph)

loss = criterion(
    prediction[target],
    y[target]
)